# DME Forecast + Denoising Low-Label on Rosbank

Runs low-label downstream classification for the forecast+denoising DME encoder on Rosbank. The notebook loads encoder weights from the best available forecast pretraining checkpoint and evaluates two modes:

- `full_finetune`: encoder and classifier are updated;
- `frozen_head`: encoder is frozen, only the classification head is trained.


In [ ]:
# Cell 1 - Setup
import json
import math
import pathlib
import pickle
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

WORKING_DIR = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
REPO_CANDIDATES = [
    WORKING_DIR / "denoising-event-sequences",
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
]
REPO_DIR = next(
    (p for p in REPO_CANDIDATES if (p / "src").exists() and (p / "scripts").exists()),
    REPO_CANDIDATES[0],
)
if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repository not found. Checked: {REPO_CANDIDATES}")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--quiet"], check=True)
sys.path.insert(0, str(REPO_DIR))

required_imports = {
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "torchmetrics": "torchmetrics",
    "matplotlib": "matplotlib",
}
missing_packages = []
for module_name, package_name in required_imports.items():
    try:
        __import__(module_name)
    except ImportError:
        missing_packages.append(package_name)
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages, "--quiet"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.forecasting import get_num_feature_dim
from src.data.splits import load_splits
from src.evaluation.classification import compute_classification_metrics
from src.models.dme_encoder import DMEEncoder
from src.training.finetune import finetune
from src.utils.logging import MetricsLogger
from src.utils.seed import set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repo   : {REPO_DIR}")
print(f"Device : {device}")
print(f"Torch  : {torch.__version__}")


In [ ]:
# Cell 2 - Runtime Knobs
PROCESSED_DIR_OVERRIDE = None
BEST_CHECKPOINT_OVERRIDE = None

OUTPUT_DIR = WORKING_DIR / "outputs" / "dme_forecast_denoising_low_label_rosbank"
LOG_DIR = OUTPUT_DIR / "logs"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for path in [OUTPUT_DIR, LOG_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RUN_FULL_FINETUNE = True
RUN_FROZEN_HEAD = True

LABEL_FRACTIONS = [0.05, 0.25, 0.50, 0.75, 1.00]
LABEL_SEEDS = [42, 43, 44, 45, 46]

FAST_DEBUG = False
if FAST_DEBUG:
    LABEL_FRACTIONS = [0.05]
    LABEL_SEEDS = [42]

METRIC_COLUMNS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]
print(f"Output dir      : {OUTPUT_DIR}")
print(f"Label fractions : {LABEL_FRACTIONS}")
print(f"Label seeds     : {LABEL_SEEDS}")


In [ ]:
# Cell 3 - Resolve Best Checkpoint And Processed Data
def _remap_path(path_value):
    if not path_value:
        return None
    path = pathlib.Path(str(path_value)).expanduser()
    if path.exists():
        return path
    raw = str(path_value)
    kaggle_prefix = "/kaggle/working/"
    if raw.startswith(kaggle_prefix):
        remapped = WORKING_DIR / raw[len(kaggle_prefix):]
        if remapped.exists():
            return remapped
    return path


def _load_summary_payloads():
    candidates = []
    for root in [WORKING_DIR / "outputs", REPO_DIR / "results"]:
        if root.exists():
            candidates.extend(root.rglob("final_forecast_pretrain_pipeline_results.json"))
    payloads = []
    for path in sorted(set(candidates)):
        try:
            payloads.append((path, json.loads(path.read_text())))
        except Exception as exc:
            print(f"Skipping unreadable summary {path}: {exc}")
    return payloads


SUMMARY_PAYLOADS = _load_summary_payloads()
print(f"Found summary payloads: {len(SUMMARY_PAYLOADS)}")


def resolve_best_checkpoint():
    if BEST_CHECKPOINT_OVERRIDE:
        ckpt = _remap_path(BEST_CHECKPOINT_OVERRIDE)
        if ckpt and ckpt.exists():
            return ckpt
        raise FileNotFoundError(f"BEST_CHECKPOINT_OVERRIDE does not exist: {BEST_CHECKPOINT_OVERRIDE}")

    for _, payload in SUMMARY_PAYLOADS:
        best_key = payload.get("best_experiment")
        experiment = payload.get("experiments", {}).get(best_key, {})
        ckpt = _remap_path(experiment.get("checkpoint"))
        if ckpt and ckpt.exists():
            return ckpt

    checkpoint_candidates = []
    for root in [WORKING_DIR / "outputs", REPO_DIR / "outputs"]:
        if root.exists():
            checkpoint_candidates.extend(root.rglob("best_forecast_checkpoint.pt"))
    checkpoint_candidates = sorted(set(checkpoint_candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    if checkpoint_candidates:
        return checkpoint_candidates[0]

    raise FileNotFoundError(
        "Could not find best_forecast_checkpoint.pt. Set BEST_CHECKPOINT_OVERRIDE "
        "to the checkpoint produced by forecast_pretrain.py."
    )


def resolve_processed_dir():
    candidates = []
    if PROCESSED_DIR_OVERRIDE:
        candidates.append(PROCESSED_DIR_OVERRIDE)
    for _, payload in SUMMARY_PAYLOADS:
        candidates.append(payload.get("processed_data"))
    candidates.extend([
        WORKING_DIR / "data" / "processed" / "rosbank_forecast_final",
        REPO_DIR / "data" / "processed" / "rosbank_forecast_final",
        REPO_DIR / "data" / "processed" / "rosbank",
    ])
    for candidate in candidates:
        path = _remap_path(candidate)
        if path and (path / "events.parquet").exists() and (path / "splits.json").exists():
            return path
    raise FileNotFoundError(
        "Could not find processed Rosbank data. Set PROCESSED_DIR_OVERRIDE "
        "to a directory with events.parquet, splits.json and preprocessor.pkl."
    )


BEST_CHECKPOINT = resolve_best_checkpoint()
PROCESSED_DIR = resolve_processed_dir()
print(f"Best checkpoint : {BEST_CHECKPOINT}")
print(f"Processed data  : {PROCESSED_DIR}")


In [ ]:
# Cell 4 - Load Config, Data And Dataloaders
set_seed(42)
checkpoint = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=False)
config = checkpoint.get("config")
if config is None:
    raise KeyError("Checkpoint does not contain a merged training config")
config = json.loads(json.dumps(config))
config.setdefault("forecasting", {})["finetune_aux_enabled"] = False

with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
    preprocessor = pickle.load(f)
splits = load_splits(PROCESSED_DIR / "splits.json")
df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")

vocab_info = checkpoint.get("vocab_info")
if vocab_info is None:
    vocab_info = {
        "event_type_vocab_size": len(preprocessor.vocab.get(preprocessor.event_type_col, {})),
        "cat_vocab_sizes": [len(preprocessor.vocab.get(col, {})) for col in preprocessor.categorical_cols],
        "num_num_features": get_num_feature_dim(preprocessor, config),
        "num_classes": int(config.get("training", {}).get("num_classes", 2)),
    }

batch_size = int(config.get("training", {}).get("batch_size", 128))
num_classes = int(vocab_info.get("num_classes", config.get("training", {}).get("num_classes", 2)))

val_loader = DataLoader(
    EventSequenceDataset(df_events, splits["val"], preprocessor, config, mode="eval"),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)
test_loader = DataLoader(
    EventSequenceDataset(df_events, splits.get("test", []), preprocessor, config, mode="eval"),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
) if splits.get("test") else None

print(f"Rows          : {len(df_events):,}")
print(f"Train entities: {len(splits['train']):,}")
print(f"Val entities  : {len(splits['val']):,}")
print(f"Test entities : {len(splits.get('test', [])):,}")
print(f"Batch size    : {batch_size}")
print(f"Vocab info    : {vocab_info}")


In [ ]:
# Cell 5 - Low-Label Helpers
def stratified_entity_subset(df, entity_ids, config, fraction, seed):
    if fraction >= 1.0:
        return list(entity_ids)
    data_cfg = config["data"]
    entity_col = data_cfg["group_col"]
    target_col = data_cfg["target_col"]
    labels = df[df[entity_col].isin(entity_ids)].groupby(entity_col)[target_col].first()
    rng = np.random.default_rng(seed)
    selected = []
    for _, group in labels.groupby(labels):
        ids = np.array(group.index.tolist())
        n = max(1, int(round(len(ids) * fraction)))
        n = min(n, len(ids))
        selected.extend(rng.choice(ids, size=n, replace=False).tolist())
    rng.shuffle(selected)
    return selected


def batch_to_device(batch, device):
    return {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}


def evaluate_on_test(model, loader, num_classes, device):
    if loader is None:
        return {}
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch_to_device(batch, device)
            probs = torch.softmax(model(batch, mode="finetune")["logits"], dim=-1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(batch["label"].cpu().numpy())
    probs_np = np.concatenate(all_probs)
    labels_np = np.concatenate(all_labels)
    return compute_classification_metrics(labels_np, probs_np, num_classes)


def aggregate_low_label(rows):
    if not rows:
        return pd.DataFrame(), []
    df = pd.DataFrame(rows)
    agg_rows = []
    for (pipeline, fraction), group in df.groupby(["pipeline", "fraction"], sort=True):
        row = {"pipeline": pipeline, "fraction": float(fraction), "n_seeds": int(group["seed"].nunique())}
        for metric in METRIC_COLUMNS:
            if metric in group:
                values = group[metric].dropna().astype(float)
                if len(values):
                    row[f"{metric}_mean"] = float(values.mean())
                    row[f"{metric}_std"] = float(values.std(ddof=0))
        agg_rows.append(row)
    return pd.DataFrame(agg_rows), agg_rows


def plot_metric(agg_df, metric):
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"
    if agg_df.empty or mean_col not in agg_df:
        return None
    fig, ax = plt.subplots(figsize=(7, 4))
    for pipeline, group in agg_df.groupby("pipeline"):
        group = group.sort_values("fraction")
        ax.errorbar(
            group["fraction"],
            group[mean_col],
            yerr=group.get(std_col),
            marker="o",
            capsize=3,
            label=pipeline,
        )
    ax.set_xlabel("Label fraction")
    ax.set_ylabel(metric)
    ax.set_title(f"DME forecast+denoising low-label: {metric}")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    out_path = FIGURES_DIR / f"{metric}_by_fraction.png"
    fig.savefig(out_path, dpi=160)
    plt.show()
    return out_path


In [ ]:
# Cell 6 - Run Low-Label Fine-Tuning
pipelines = []
if RUN_FULL_FINETUNE:
    pipelines.append(("full_finetune", False))
if RUN_FROZEN_HEAD:
    pipelines.append(("frozen_head", True))

all_rows = []

for pipeline_name, frozen_encoder in pipelines:
    for fraction in LABEL_FRACTIONS:
        for seed in LABEL_SEEDS:
            set_seed(seed)
            train_ids = stratified_entity_subset(df_events, splits["train"], config, fraction, seed)
            run_id = f"{pipeline_name}_f{fraction:.2f}_s{seed}"
            run_dir = OUTPUT_DIR / pipeline_name / f"f{fraction:.2f}_seed{seed}"
            run_dir.mkdir(parents=True, exist_ok=True)

            print(f"[{run_id}] train_entities={len(train_ids)} / {len(splits['train'])}")
            train_loader = DataLoader(
                EventSequenceDataset(df_events, train_ids, preprocessor, config, mode="finetune"),
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_fn,
                num_workers=0,
            )

            model = DMEEncoder(config, vocab_info)
            logger = MetricsLogger(str(run_dir / "logs"), run_id)
            best_finetune_checkpoint = finetune(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                config=config,
                output_dir=str(run_dir / "checkpoints"),
                device=device,
                logger=logger,
                pretrained_checkpoint=str(BEST_CHECKPOINT),
                frozen_encoder=frozen_encoder,
                label_fraction=1.0,
                vocab_info=vocab_info,
            )

            state = torch.load(best_finetune_checkpoint, map_location="cpu", weights_only=False)
            model.load_state_dict(state["model_state_dict"])
            model.to(device)
            test_metrics = evaluate_on_test(model, test_loader, num_classes, device)
            row = {
                "pipeline": pipeline_name,
                "fraction": float(fraction),
                "seed": int(seed),
                "n_train_entities": int(len(train_ids)),
                "checkpoint": str(best_finetune_checkpoint),
                **test_metrics,
            }
            all_rows.append(row)
            with (run_dir / "test_metrics.json").open("w") as f:
                json.dump(row, f, indent=2)
            print({k: row[k] for k in ["pipeline", "fraction", "seed", "roc_auc", "pr_auc", "macro_f1"] if k in row})

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


In [ ]:
# Cell 7 - Save Aggregates
all_runs_df = pd.DataFrame(all_rows)
aggregate_df, aggregate_rows = aggregate_low_label(all_rows)

all_runs_csv = METRICS_DIR / "low_label_all_runs.csv"
all_runs_json = METRICS_DIR / "low_label_all_runs.json"
aggregate_csv = METRICS_DIR / "low_label_aggregate.csv"
aggregate_json = METRICS_DIR / "low_label_aggregate.json"

all_runs_df.to_csv(all_runs_csv, index=False)
all_runs_df.to_json(all_runs_json, orient="records", indent=2)
aggregate_df.to_csv(aggregate_csv, index=False)
with aggregate_json.open("w") as f:
    json.dump(aggregate_rows, f, indent=2)

print(f"Saved per-run CSV   : {all_runs_csv}")
print(f"Saved aggregate CSV : {aggregate_csv}")

try:
    display(aggregate_df)
except NameError:
    print(aggregate_df.to_string(index=False))


In [ ]:
# Cell 8 - Plots
plot_paths = {}
for metric in ["roc_auc", "pr_auc", "macro_f1"]:
    path = plot_metric(aggregate_df, metric)
    if path is not None:
        plot_paths[metric] = str(path)
print(json.dumps(plot_paths, indent=2))
